# Tugas Akhir

## Prediksi Harga Mobil dengan Regresi

Sebuah perusahaan otomotif ingin memahami faktor yang menentukan harga jual mobil dan seberapa akurat harga itu bisa diperkirakan dari spesifikasi teknisnya. Notebook ini menjawab dua pertanyaan tersebut secara berurutan: memilih fitur yang relevan, membangun model regresi, lalu menilai kualitasnya pada data yang belum pernah dilihat model.

### Tujuan

- memuat dan memeriksa kualitas data;
- menyiapkan fitur numerik dan kategorikal dalam satu pipeline;
- melatih model regresi dan mengevaluasinya dengan RMSE, MAE, dan R kuadrat;
- menafsirkan hasil dalam konteks bisnis.

## 1. Memuat data

Dataset berisi 205 model mobil dengan 26 atribut, mencakup dimensi, kapasitas mesin, konsumsi bahan bakar, dan harga jual (`price`) sebagai target.

In [ ]:
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

data_path = Path("data/CarPrice_Assignment.csv")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/CarPrice_Assignment.csv"

if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(data_url, data_path)

cars = pd.read_csv(data_path)
print("Ukuran data:", cars.shape)
cars.head()

Data terdiri atas 205 baris dan 26 kolom. Kolom `price` adalah target, sisanya calon prediktor dengan campuran tipe numerik dan teks. Pemeriksaan awal menunjukkan tidak ada nilai kosong, dan setiap `car_ID` unik sehingga tidak ada baris ganda.

In [ ]:
print("Nilai kosong per kolom:", int(cars.isnull().sum().sum()))
print("car_ID duplikat:", int(cars["car_ID"].duplicated().sum()))
print(cars["price"].describe())

## 2. Menyiapkan fitur

`CarName` memuat merek dan model sekaligus. Untuk pemodelan, hanya merek yang dipakai, dan beberapa salah ketik diseragamkan. Kolom pengenal seperti `car_ID` serta `symboling` (peringkat risiko asuransi) tidak relevan untuk prediksi harga sehingga dibuang.

Kolom kategorikal di-encode dengan `OneHotEncoder`, kolom numerik distandarkan. Keduanya dirangkai dalam `ColumnTransformer` agar konsisten antara data latih dan data uji.

In [ ]:
cars["brand"] = cars["CarName"].str.split(" ").str[0].str.lower()
cars["brand"] = cars["brand"].replace({
    "vw": "volkswagen", "vokswagen": "volkswagen",
    "maxda": "mazda", "porcshnye": "porsche", "porcshce": "porsche",
    "toyouta": "toyota",
})

model_data = cars.drop(columns=["car_ID", "symboling", "CarName"])

numeric_features = model_data.select_dtypes(include="number").columns.drop("price").tolist()
categorical_features = model_data.select_dtypes(exclude="number").columns.tolist()

print("Fitur numerik :", numeric_features)
print("Fitur kategorikal:", categorical_features)

Sebelum pemodelan, korelasi antarfitur numerik dan target diperiksa. Fitur dengan korelasi tinggi terhadap harga adalah kandidat kuat, sedangkan pasangan fitur yang saling berkorelasi tinggi berisiko menimbulkan multikolinearitas.

In [ ]:
correlation = model_data.corr(numeric_only=True)["price"].sort_values(ascending=False)
correlation

In [ ]:
plt.figure(figsize=(10, 4))
correlation.drop("price").plot(kind="bar")
plt.ylabel("Korelasi dengan harga")
plt.tight_layout()
plt.show()

`enginesize`, `curbweight`, `horsepower`, dan `carwidth` berkorelasi paling kuat dengan harga. Keempatnya menggambarkan ukuran dan tenaga mobil, sehingga wajar bila harganya ikut naik. Sebaliknya, `citympg` dan `highwaympg` berkorelasi negatif: mobil irit bahan bakar cenderung berharga lebih rendah pada dataset ini.

## 3. Membangun model

Data dibagi 80% latih dan 20% uji. Model yang dipakai adalah Ridge dengan alpha kecil, pilihan yang wajar karena fitur numerik saling berkorelasi sehingga regresi linear biasa rawan koefisien tidak stabil.

In [ ]:
features = model_data.drop(columns="price")
target = model_data["price"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(transformers=[
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numeric", StandardScaler(), numeric_features),
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regression", Ridge(alpha=1.0)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Jumlah data latih:", len(X_train))
print("Jumlah data uji  :", len(X_test))

## 4. Evaluasi model

Tiga ukuran dipakai bersama karena masing-masing punya kelemahan bila berdiri sendiri. RMSE dan MAE berada dalam satuan dolar sehingga mudah ditafsirkan, sedangkan R kuadrat menunjukkan proporsi variasi harga yang berhasil dijelaskan model.

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"RMSE: {rmse:,.0f}")
print(f"MAE : {mae:,.0f}")
print(f"R2  : {r2:.4f}")

MAE lebih kecil daripada RMSE, tanda ada sejumlah mobil yang harganya jauh di luar pola umum sehingga galatnya besar. Selisih keduanya bukan kesalahan hitung, melainkan akibat RMSE mengkuadratkan galat sebelum diakar.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predictions, alpha=0.7)
limit = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
plt.plot(limit, limit, color="red", ls="--", label="prediksi sempurna")
plt.xlabel("Harga aktual")
plt.ylabel("Harga prediksi")
plt.legend()
plt.tight_layout()
plt.show()

Titik-titik mengumpul di sekitar garis diagonal, tanda model menangkap pola utama harga. Penyimpangan terbesar muncul pada mobil berharga sangat tinggi, yang jumlahnya sedikit di data sehingga sulit dipelajari model.

## 5. Menafsirkan hasil

Koefisien model menunjukkan arah pengaruh setiap fitur setelah skala diseragamkan. Nilai positif berarti fitur menaikkan harga, nilai negatif menurunkannya.

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["regression"].coef_

importance = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
importance["abs_coefficient"] = importance["coefficient"].abs()
importance.sort_values("abs_coefficient", ascending=False).head(12)

Koefisien terbesar justru dimiliki fitur merek dan posisi mesin. BMW, Buick, dan Porsche menaikkan harga sekitar 3.400 sampai 4.300 dolar dibanding merek acuan, sedangkan mesin di belakang (`enginelocation_rear`) menambah sekitar 3.400 dolar. Di antara fitur teknis, `enginesize` dan `curbweight` adalah pendorong harga terkuat. Sebaliknya, mesin tipe OHCV dan beberapa merek berkoefisien negatif.

Temuan ini masuk akal: merek membawa reputasi dan segmen pasar yang tidak tertangkap angka spesifikasi, sementara ukuran mesin dan bobot mencerminkan kelas kendaraan. Model karenanya berguna untuk menjelaskan pola harga, bukan sebagai patokan tunggal.

### Catatan keterbatasan

- Dataset hanya 205 mobil, sehingga beberapa titik ekstrem mudah menggeser koefisien.
- Fitur ukuran saling berkorelasi kuat, dan regularisasi hanya meredamnya sebagian.
- Harga mobil juga dipengaruhi faktor di luar data, misalnya kondisi pasar dan merek pesaing.

## Ringkasan

Model Ridge dengan fitur numerik dan kategorikal mencapai R kuadrat sekitar 0,9 pada data uji, dengan galat rata-rata beberapa ribu dolar. Ukuran mesin, bobot, dan tenaga kuda adalah penentu harga utama, sementara efisiensi bahan bakar berhubungan negatif.